# TCGA-KIRC epithelial subtype ssGSEA

Calculate epithelial subtype scores from the top and bottom DEG signatures.

In [ ]:
import numpy as np
import pandas as pd
import gseapy as gp

In [ ]:
#Define a dictionary to store the results
top_100_cell_type_markers = {}

# Read the CSV files and store the top 100 genes in the dictionary

top_100_cell_type_markers['Epi_ALDOB'] = pd.read_csv('../cellsubtype_degs/Epi_ALDOB_degs_epi.csv', index_col=0, nrows=100).index.tolist()
top_100_cell_type_markers['Epi_AQP2'] = pd.read_csv('../cellsubtype_degs/Epi_AQP2_degs_epi.csv', index_col=0, nrows=100).index.tolist()
top_100_cell_type_markers['Epi_CA12'] = pd.read_csv('../cellsubtype_degs/Epi_CA12_degs_epi.csv', index_col=0, nrows=100).index.tolist()
#top_100_cell_type_markers['Epi_CD45'] = pd.read_csv('deg_Epi_CD45.csv', index_col=0, nrows=100).index.tolist()
top_100_cell_type_markers['Epi_CA9'] = pd.read_csv('../cellsubtype_degs/Epi_CA9_degs_epi.csv', index_col=0, nrows=100).index.tolist()
top_100_cell_type_markers['Epi_GPX3'] = pd.read_csv('../cellsubtype_degs/Epi_GPX3_degs_epi.csv', index_col=0, nrows=100).index.tolist()
top_100_cell_type_markers['Epi_JUN'] = pd.read_csv('../cellsubtype_degs/Epi_JUN_degs_epi.csv', index_col=0, nrows=100).index.tolist()
top_100_cell_type_markers['Epi_MIOX'] = pd.read_csv('../cellsubtype_degs/Epi_MIOX_degs_epi.csv', index_col=0, nrows=100).index.tolist()
top_100_cell_type_markers['Epi_VIM'] = pd.read_csv('../cellsubtype_degs/Epi_VIM_degs_epi.csv', index_col=0, nrows=100).index.tolist()



In [ ]:
#Define a dictionary to store the results
btm_100_cell_type_markers = {}

# Read the CSV files and store the btm 100 genes in the dictionary

btm_100_cell_type_markers['Epi_ALDOB'] = pd.read_csv('../cellsubtype_degs/Epi_ALDOB_degs_epi.csv', index_col=0).tail(100).index.tolist()
btm_100_cell_type_markers['Epi_AQP2'] = pd.read_csv('../cellsubtype_degs/Epi_AQP2_degs_epi.csv', index_col=0).tail(100).index.tolist()
btm_100_cell_type_markers['Epi_CA12'] = pd.read_csv('../cellsubtype_degs/Epi_CA12_degs_epi.csv', index_col=0).tail(100).index.tolist()
#btm_100_cell_type_markers['Epi_CD45'] = pd.read_csv('deg_Epi_CD45.csv', index_col=0).index.tolist()
btm_100_cell_type_markers['Epi_CA9'] = pd.read_csv('../cellsubtype_degs/Epi_CA9_degs_epi.csv', index_col=0).tail(100).index.tolist()
btm_100_cell_type_markers['Epi_GPX3'] = pd.read_csv('../cellsubtype_degs/Epi_GPX3_degs_epi.csv', index_col=0).tail(100).index.tolist()
btm_100_cell_type_markers['Epi_JUN'] = pd.read_csv('../cellsubtype_degs/Epi_JUN_degs_epi.csv', index_col=0).tail(100).index.tolist()
btm_100_cell_type_markers['Epi_MIOX'] = pd.read_csv('../cellsubtype_degs/Epi_MIOX_degs_epi.csv', index_col=0).tail(100).index.tolist()
btm_100_cell_type_markers['Epi_VIM'] = pd.read_csv('../cellsubtype_degs/Epi_VIM_degs_epi.csv', index_col=0).tail(100).index.tolist()



In [ ]:
btm_100_cell_type_markers

In [ ]:
tcga = pd.read_csv('gene_symbol_fpkm_star_kirc.csv',index_col=0)
tcga

In [ ]:
top_100_ss = gp.ssgsea(data=tcga,
               gene_sets=top_100_cell_type_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)

In [ ]:
top_100_nes = top_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
top_100_nes_t = top_100_nes.T
top_100_nes_t.to_csv('nes_top_100.csv')

In [ ]:
btm_100_ss = gp.ssgsea(data=tcga,
               gene_sets=btm_100_cell_type_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)


In [ ]:
btm_100_nes = btm_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
btm_100_nes_t = btm_100_nes.T
btm_100_nes_t.to_csv('nes_btm_100.csv')

In [ ]:
nes_diff = top_100_nes_t-btm_100_nes_t
nes_diff

In [ ]:
nes_diff.to_csv('nes_diff.csv')

# TCGA-KIRC overall survival analysis

Compare the top and bottom score quartiles for each epithelial subtype.

In [ ]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch
from scipy.stats import chi2

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["text.usetex"] = False

# Use the directory from which this notebook is executed.
BASE_DIR = Path.cwd()
SCORE_PATH = BASE_DIR / "nes_diff.csv"
SURVIVAL_PATH = BASE_DIR / "TCGA-KIRC.survival.csv"
OUT_DIR = BASE_DIR / "tcga_epi_only_km_cm_epi_style_cm_epi_axispad_quartile_only"

PALETTE = {
    "High": "#d62728",
    "Low": "#1f77b4",
    "Top 25%": "#d62728",
    "Bottom 25%": "#1f77b4",
}


def clean_filename(value):
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    return value.strip("_")


def format_pvalue(p_value):
    if not np.isfinite(p_value):
        return "NA"
    if p_value < 1e-4:
        return f"{p_value:.2e}"
    return f"{p_value:.4f}"


def km_confidence_interval(survival, greenwood):
    if survival <= 0:
        return 0.0, 0.0
    if not np.isfinite(greenwood):
        return 0.0, 1.0
    se_log = np.sqrt(max(greenwood, 0.0))
    z = 1.959963984540054
    lower = survival * np.exp(-z * se_log)
    upper = survival * np.exp(z * se_log)
    return float(max(0.0, lower)), float(min(1.0, upper))


def km_estimate(times, events):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    if len(times) == 0:
        return np.array([0.0]), np.array([1.0]), np.array([1.0]), np.array([1.0]), np.array([]), np.array([])

    event_times = np.sort(np.unique(times[events == 1]))
    x = [0.0]
    y = [1.0]
    lower = [1.0]
    upper = [1.0]
    survival = 1.0
    greenwood = 0.0
    survival_after_event = {}

    for t in event_times:
        n_risk = np.sum(times >= t)
        n_events = np.sum((times == t) & (events == 1))
        if n_risk <= 0:
            continue
        survival *= 1.0 - n_events / n_risk
        if n_risk > n_events:
            greenwood += n_events / (n_risk * (n_risk - n_events))
        ci_lower, ci_upper = km_confidence_interval(survival, greenwood)
        x.extend([float(t), float(t)])
        y.extend([y[-1], float(survival)])
        lower.extend([lower[-1], ci_lower])
        upper.extend([upper[-1], ci_upper])
        survival_after_event[float(t)] = float(survival)

    max_time = float(np.max(times))
    if x[-1] < max_time:
        x.append(max_time)
        y.append(float(survival))
        lower.append(float(lower[-1]))
        upper.append(float(upper[-1]))

    censor_times = times[events == 0]
    censor_y = []
    for ct in censor_times:
        s = 1.0
        for et in event_times[event_times <= ct]:
            s = survival_after_event.get(float(et), s)
        censor_y.append(s)

    return (
        np.asarray(x),
        np.asarray(y),
        np.asarray(lower),
        np.asarray(upper),
        np.asarray(censor_times),
        np.asarray(censor_y),
    )


def logrank_pvalue(df, time_col, event_col, group_col, groups):
    data = df[df[group_col].isin(groups)].copy()
    if data[group_col].nunique() < 2:
        return np.nan

    event_times = np.sort(data.loc[data[event_col] == 1, time_col].unique())
    observed = np.zeros(len(groups), dtype=float)
    expected = np.zeros(len(groups), dtype=float)
    variance = np.zeros((len(groups), len(groups)), dtype=float)
    times = data[time_col].to_numpy(dtype=float)
    events = data[event_col].to_numpy(dtype=int)
    group_values = data[group_col].to_numpy()

    for t in event_times:
        at_risk = times >= t
        at_event = (times == t) & (events == 1)
        n_total = int(at_risk.sum())
        d_total = int(at_event.sum())
        if n_total <= 1 or d_total == 0:
            continue

        n_by_group = np.asarray([np.sum(at_risk & (group_values == g)) for g in groups], dtype=float)
        d_by_group = np.asarray([np.sum(at_event & (group_values == g)) for g in groups], dtype=float)
        p = n_by_group / n_total
        observed += d_by_group
        expected += d_total * p
        factor = d_total * (n_total - d_total) / (n_total - 1)
        variance += factor * (np.diag(p) - np.outer(p, p))

    diff = observed[:-1] - expected[:-1]
    var = variance[:-1, :-1]
    try:
        statistic = float(diff.T @ np.linalg.pinv(var) @ diff)
    except np.linalg.LinAlgError:
        return np.nan
    return float(chi2.sf(statistic, len(groups) - 1))


def get_time_ticks(max_time, break_by):
    upper = break_by * np.ceil(max_time / break_by)
    ticks = np.arange(0, upper + 0.5 * break_by, break_by)
    return ticks, float(upper)


def add_step_confidence_band(ax, x, lower, upper, color, alpha=0.15):
    if len(x) < 2:
        return
    verts = []
    codes = []
    for i, (xx, yy) in enumerate(zip(x, upper)):
        codes.append(MplPath.MOVETO if i == 0 else MplPath.LINETO)
        verts.append((xx, yy))
    for xx, yy in zip(x[::-1], lower[::-1]):
        codes.append(MplPath.LINETO)
        verts.append((xx, yy))
    codes.append(MplPath.CLOSEPOLY)
    verts.append((x[0], upper[0]))
    patch = PathPatch(MplPath(verts, codes), facecolor=color, alpha=alpha, edgecolor="none", zorder=1)
    ax.add_patch(patch)


def draw_risk_table(risk_ax, current_df, time_col, group_col, groups, colors, labels, ticks, x_label, x_left, x_upper):
    n_rows = len(groups)
    risk_ax.set_xlim(x_left, x_upper)
    risk_ax.set_xticks(ticks)
    risk_ax.set_ylim(-0.75, n_rows + 1.2)
    risk_ax.set_yticks([n_rows - row_idx - 0.5 for row_idx in range(n_rows)])
    risk_ax.set_yticklabels(labels, fontsize=7.0)
    for tick_label, color in zip(risk_ax.get_yticklabels(), colors):
        tick_label.set_color(color)
        tick_label.set_fontweight("bold")

    risk_ax.spines["left"].set_visible(True)
    risk_ax.spines["bottom"].set_visible(True)
    risk_ax.spines["left"].set_linewidth(0.8)
    risk_ax.spines["bottom"].set_linewidth(0.8)
    risk_ax.spines["right"].set_visible(False)
    risk_ax.spines["top"].set_visible(False)
    risk_ax.grid(False)
    risk_ax.tick_params(axis="x", labelsize=8, colors="black", length=3, width=0.8)
    risk_ax.tick_params(axis="y", length=0, pad=8)
    risk_ax.set_xlabel(x_label, fontsize=10)
    risk_ax.text(x_left, n_rows + 0.85, "Number at risk", ha="left", va="center", fontsize=9, fontweight="bold")

    for row_idx, group in enumerate(groups):
        group_df = current_df[current_df[group_col] == group]
        y_pos = n_rows - row_idx - 0.5
        for tick in ticks:
            at_risk = int((group_df[time_col] >= tick).sum())
            ha = "left" if np.isclose(tick, x_left) else "center"
            risk_ax.text(tick, y_pos, str(at_risk), ha=ha, va="center", fontsize=7, color="black")


def plot_km_with_risk_table(current_df, time_col, event_col, group_col, order, colors, title, x_label, y_label, break_by, output_base):
    present_groups = [g for g in order if g in set(current_df[group_col])]
    valid_groups = []
    for group in present_groups:
        group_df = current_df[current_df[group_col] == group]
        if len(group_df) > 0 and group_df[time_col].notna().all() and group_df[event_col].notna().all():
            valid_groups.append(group)
    if len(valid_groups) < 2:
        return None

    max_time = float(current_df[time_col].max())
    ticks, x_upper = get_time_ticks(max_time, break_by)
    zero_pad_fraction = 0.035
    x_left = -(zero_pad_fraction / (1.0 - zero_pad_fraction)) * x_upper
    y_top = 1.03
    y_bottom = -(zero_pad_fraction / (1.0 - zero_pad_fraction)) * y_top

    fig = plt.figure(figsize=(5.0, 4.8), constrained_layout=False)
    grid = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3.2, 1.05], hspace=0.08)
    ax = fig.add_subplot(grid[0])
    risk_ax = fig.add_subplot(grid[1])

    plotted_groups = []
    plotted_colors = []
    plotted_labels = []
    for group in order:
        if group not in valid_groups:
            continue
        group_df = current_df[current_df[group_col] == group]
        color = colors[group]
        x, y, ci_lower, ci_upper, censor_x, censor_y = km_estimate(group_df[time_col], group_df[event_col])
        add_step_confidence_band(ax, x, ci_lower, ci_upper, color)
        ax.step(x, y, where="post", color=color, linewidth=1.6, label=f"{group}: {len(group_df)}", zorder=2)
        if len(censor_x) > 0:
            ax.plot(censor_x, censor_y, linestyle="None", marker="+", markersize=4.5, markeredgewidth=0.9, color=color, zorder=3)
        plotted_groups.append(group)
        plotted_colors.append(color)
        plotted_labels.append(group)

    p_value = logrank_pvalue(current_df, time_col, event_col, group_col, plotted_groups)
    ax.text(0.04, 0.08, f"log-rank p = {format_pvalue(p_value)}", transform=ax.transAxes, fontsize=9, ha="left", va="bottom")

    ax.set_xlim(x_left, x_upper)
    ax.set_ylim(y_bottom, y_top)
    ax.set_xticks(ticks)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.tick_params(axis="x", labelbottom=False)
    ax.set_ylabel(y_label, fontsize=10)
    ax.set_title(title, fontsize=10, pad=7)
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    draw_risk_table(risk_ax, current_df, time_col, group_col, plotted_groups, plotted_colors, plotted_labels, ticks, x_label, x_left, x_upper)
    fig.subplots_adjust(left=0.18, right=0.98, top=0.92, bottom=0.13)

    output_base = Path(output_base)
    output_base.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_base.with_suffix(".pdf"), dpi=300, bbox_inches="tight")
    fig.savefig(output_base.with_suffix(".svg"), dpi=300, bbox_inches="tight")
    plt.close(fig)

    return {
        "n_samples": int(len(current_df)),
        "n_groups": int(len(plotted_groups)),
        "logrank_p": p_value,
        **{f"n_{group}": int((current_df[group_col] == group).sum()) for group in plotted_groups},
    }


def load_inputs():
    scores = pd.read_csv(SCORE_PATH, index_col=0)
    scores.index = scores.index.astype(str).str.replace(".", "-", regex=False)
    scores = scores.groupby(scores.index).mean()

    survival = pd.read_csv(SURVIVAL_PATH, index_col=0)
    survival = survival[["OS.time", "OS"]].copy()
    survival["OS.time"] = pd.to_numeric(survival["OS.time"], errors="coerce")
    survival["OS"] = pd.to_numeric(survival["OS"], errors="coerce")
    survival = survival.dropna(subset=["OS.time", "OS"])
    survival = survival[survival["OS.time"] > 0].copy()
    survival["OS"] = survival["OS"].astype(int)
    survival = survival.groupby(survival.index.astype(str)).first()

    common = scores.index.intersection(survival.index)
    scores = scores.loc[common].copy()
    survival = survival.loc[common].copy()
    return scores, survival


def run_grouping(scores, survival, grouping):
    rows = []
    out_subdir = OUT_DIR / grouping
    out_subdir.mkdir(parents=True, exist_ok=True)
    epi_cols = [c for c in scores.columns if c.startswith("Epi_")]

    for feature in epi_cols:
        current = survival.join(scores[[feature]], how="inner").dropna(subset=["OS.time", "OS", feature]).copy()
        if grouping == "median":
            cutoff = float(current[feature].median())
            current["group"] = np.where(current[feature] >= cutoff, "High", "Low")
            order = ["High", "Low"]
            title = f"TCGA-KIRC OS: {feature} median split"
            extra = {"median": cutoff}
        elif grouping == "quartile":
            q1 = float(current[feature].quantile(0.25))
            q3 = float(current[feature].quantile(0.75))
            low = current[current[feature] <= q1].copy()
            high = current[current[feature] >= q3].copy()
            low["group"] = "Bottom 25%"
            high["group"] = "Top 25%"
            current = pd.concat([high, low], axis=0)
            order = ["Top 25%", "Bottom 25%"]
            title = f"TCGA-KIRC OS: {feature} top vs bottom quartile"
            extra = {"q1": q1, "q3": q3}
        else:
            raise ValueError(grouping)

        result = plot_km_with_risk_table(
            current_df=current,
            time_col="OS.time",
            event_col="OS",
            group_col="group",
            order=order,
            colors=PALETTE,
            title=title,
            x_label="Days",
            y_label="Overall survival",
            break_by=1000,
            output_base=out_subdir / f"TCGA_KIRC_OS_{clean_filename(feature)}_{grouping}",
        )
        if result is None:
            continue
        rows.append({"feature": feature, "grouping": grouping, **extra, **result})

    summary = pd.DataFrame(rows)
    summary.to_csv(out_subdir / f"tcga_epi_only_{grouping}_km_summary.csv", index=False)
    return summary


def main():
    scores, survival = load_inputs()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest = {
        "score_path": str(SCORE_PATH),
        "survival_path": str(SURVIVAL_PATH),
        "matched_samples": int(len(scores)),
        "events": int(survival["OS"].sum()),
        "censored": int((survival["OS"] == 0).sum()),
        "features": [c for c in scores.columns if c.startswith("Epi_")],
    }
    pd.Series(manifest, dtype="object").to_csv(OUT_DIR / "input_manifest.csv")

    summary = run_grouping(scores, survival, "quartile")
    summary.to_csv(OUT_DIR / "tcga_epi_only_quartile_only_km_summary.csv", index=False)
    print(f"Done. Output: {OUT_DIR}")
    print(f"Matched samples: {manifest['matched_samples']}, events: {manifest['events']}, censored: {manifest['censored']}")
    print(f"Features: {', '.join(manifest['features'])}")


if __name__ == "__main__":
    main()
